# Verify `voxelize_mesh_gpu` vs Open3D

This notebook validates `o_voxel.convert.voxelize_mesh_gpu` against Open3D voxelization:

1. Choose whether to JIT compile the current local sources or directly import an already-installed `o_voxel` package.
2. Load the selected Python API.
3. Generate synthetic Gaussian triangle soup directly in the notebook, including a few degenerate triangles.
4. Compare occupied voxels against Open3D and report the Jaccard score.

No local mesh file is loaded.


In [1]:
import os
import sys
import time
import types
import importlib
import importlib.util
from pathlib import Path

import numpy as np
import torch
from torch.utils.cpp_extension import load

import open3d as o3d


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
ROOT = Path(r'/mnt/nvmefs/Projects/Part Generation/TRELLIS.2-o-voxel-gpu-mod/o-voxel').resolve()
USE_JIT = False
INSTALLED_IMPORT_NAME = 'o_voxel'

print('ROOT =', ROOT)
print('USE_JIT =', USE_JIT)
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device =', torch.cuda.get_device_name(0))


def build_jit_extension():
    sources = [
    'src/hash/hash.cu',
    'src/convert/flexible_dual_grid.cpp',
    'src/convert/volumetic_attr.cpp',
    'src/convert/mesh_to_flexible_dual_grid_gpu/torch_bindings.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/flexible_dual_grid_gpu.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/intersection_qef.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/voxelize_mesh_oct.cu',
    'src/convert/mesh_to_flexible_dual_grid_gpu/voxel_traverse_edge_dda.cu',
    'src/serialize/api.cu',
    'src/serialize/hilbert.cu',
    'src/serialize/z_order.cu',
    'src/io/svo.cpp',
    'src/io/filter_parent.cpp',
    'src/io/filter_neighbor.cpp',
    'src/rasterize/rasterize.cu',
    'src/ext.cpp',
]
    full_sources = [str(ROOT / s) for s in sources]
    missing = [s for s in full_sources if not Path(s).exists()]
    if missing:
        raise FileNotFoundError(f'Missing sources: {missing}')

    build_dir = ROOT / '.verify_build'
    build_dir.mkdir(parents=True, exist_ok=True)

    unique_suffix = f"{os.getpid()}_{time.time_ns()}_{os.urandom(4).hex()}"
    mod_name = f"o_voxel_verify_{unique_suffix}"

    max_jobs = max(1, os.cpu_count() or 1)
    os.environ['MAX_JOBS'] = str(max_jobs)
    print('MAX_JOBS =', os.environ['MAX_JOBS'])
    print('JIT module name =', mod_name)

    ext_mod = load(
        name=mod_name,
        sources=full_sources,
        extra_include_paths=[str(ROOT / 'third_party/eigen')],
        extra_cflags=['-O3', '-std=c++17'],
        extra_cuda_cflags=['-O3', '-std=c++17', '--expt-relaxed-constexpr'],
        with_cuda=True,
        build_directory=str(build_dir),
        verbose=True,
    )
    print('JIT build/link: OK')
    print('jit module path =', ext_mod.__file__)
    return ext_mod


def load_local_flexible_dual_grid(ext_mod):
    pkg = types.ModuleType('o_voxel')
    pkg.__path__ = [str(ROOT / 'o_voxel')]
    pkg._C = ext_mod
    sys.modules['o_voxel'] = pkg
    sys.modules['o_voxel._C'] = ext_mod

    convert_pkg = types.ModuleType('o_voxel.convert')
    convert_pkg.__path__ = [str(ROOT / 'o_voxel' / 'convert')]
    sys.modules['o_voxel.convert'] = convert_pkg

    spec = importlib.util.spec_from_file_location(
        'o_voxel.convert.flexible_dual_grid',
        ROOT / 'o_voxel' / 'convert' / 'flexible_dual_grid.py',
    )
    mod = importlib.util.module_from_spec(spec)
    sys.modules['o_voxel.convert.flexible_dual_grid'] = mod
    spec.loader.exec_module(mod)
    return mod


if USE_JIT:
    ext_mod = build_jit_extension()
    fdg_api = load_local_flexible_dual_grid(ext_mod)
    api_mode = 'jit'
else:
    installed_pkg = importlib.import_module(INSTALLED_IMPORT_NAME)
    ext_mod = installed_pkg._C
    fdg_api = importlib.import_module(f'{INSTALLED_IMPORT_NAME}.convert.flexible_dual_grid')
    api_mode = 'installed'
    print('installed package path =', getattr(installed_pkg, '__file__', '<package>'))
    print('installed extension path =', getattr(ext_mod, '__file__', '<extension>'))
    print('installed API path =', getattr(fdg_api, '__file__', '<module>'))

print('api_mode =', api_mode)
print('ext_mod =', ext_mod)
print('fdg_api =', fdg_api)


ROOT = /mnt/nvmefs/Projects/Part Generation/TRELLIS.2-o-voxel-gpu-mod/o-voxel
USE_JIT = False
torch = 2.6.0+cu124
cuda available = True
cuda device = NVIDIA GeForce RTX 4090
installed package path = /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/__init__.py
installed extension path = /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/_C.cpython-310-x86_64-linux-gnu.so
installed API path = /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/convert/flexible_dual_grid.py
api_mode = installed
ext_mod = <module 'o_voxel._C' from '/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/_C.cpython-310-x86_64-linux-gnu.so'>
fdg_api = <module 'o_voxel.convert.flexible_dual_grid' from '/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/o_voxel/convert/flexible_dual_grid.py'>


In [4]:
print('has mesh_to_flexible_dual_grid_cpu =', hasattr(ext_mod, 'mesh_to_flexible_dual_grid_cpu'))
print('has mesh_to_flexible_dual_grid_gpu =', hasattr(ext_mod, 'mesh_to_flexible_dual_grid_gpu'))
print('has intersection_occ =', hasattr(fdg_api, 'intersection_occ'))
print('has intersect_qef =', hasattr(fdg_api, 'intersect_qef'))
print('has voxelize_mesh_gpu =', hasattr(fdg_api, 'voxelize_mesh_gpu'))
print('has voxelize_edge_gpu =', hasattr(fdg_api, 'voxelize_edge_gpu'))
print('has face_qef =', hasattr(fdg_api, 'face_qef'))
print('has voxel_traverse_edge_dda_gpu =', hasattr(fdg_api, 'voxel_traverse_edge_dda_gpu'))
print('has boundary_qef =', hasattr(fdg_api, 'boundary_qef'))


has mesh_to_flexible_dual_grid_cpu = True
has mesh_to_flexible_dual_grid_gpu = True
has intersection_occ = True
has intersect_qef = True
has voxelize_mesh_gpu = True
has voxelize_edge_gpu = True
has face_qef = True
has voxel_traverse_edge_dda_gpu = True
has boundary_qef = True


In [5]:
assert torch.cuda.is_available(), 'CUDA is required for this notebook.'
device = torch.device('cuda:0')
torch.cuda.set_device(device)

GRID = 128
N_VERT = 8000
N_FACE = 24000
AABB = torch.tensor([[0.0, 0.0, 0.0], [1.0, 1.0, 1.0]], dtype=torch.float32, device=device)

torch.manual_seed(7)
vertices = (0.5 + 0.18 * torch.randn(N_VERT, 3, device=device, dtype=torch.float32)).clamp_(0.0, 1.0)
vertices[1] = vertices[0]
vertices[3] = vertices[2]

faces = torch.randint(0, N_VERT, (N_FACE, 3), device=device, dtype=torch.int64)
faces[:3] = torch.tensor([[0, 0, 1], [2, 3, 3], [4, 4, 4]], device=device, dtype=torch.int64)
faces = faces.to(torch.int32).contiguous()

voxel_size = ((AABB[1] - AABB[0]) / GRID).to(torch.float32).contiguous()
grid_range = torch.tensor([[0, 0, 0], [GRID, GRID, GRID]], device=device, dtype=torch.int32)

print('vertices:', vertices.shape, vertices.dtype, vertices.device)
print('faces   :', faces.shape, faces.dtype, faces.device)
print('voxel_size:', voxel_size)
print('grid_range:', grid_range)


vertices: torch.Size([8000, 3]) torch.float32 cuda:0
faces   : torch.Size([24000, 3]) torch.int32 cuda:0
voxel_size: tensor([0.0078, 0.0078, 0.0078], device='cuda:0')
grid_range: tensor([[  0,   0,   0],
        [128, 128, 128]], device='cuda:0', dtype=torch.int32)


In [6]:
face_id, voxel_ijk = fdg_api.voxelize_mesh_gpu(
    vertices,
    faces,
    voxel_size=voxel_size,
    grid_range=grid_range,
)
ours_unique = torch.unique(voxel_ijk.to(torch.int32), dim=0)
print('face_id:', face_id.shape, face_id.dtype, face_id.device)
print('voxel_ijk:', voxel_ijk.shape, voxel_ijk.dtype, voxel_ijk.device)
print('ours unique voxel count:', ours_unique.shape[0])


face_id: torch.Size([36236356]) torch.int32 cuda:0
voxel_ijk: torch.Size([36236356, 3]) torch.int32 cuda:0
ours unique voxel count: 1049126


In [7]:
mesh = o3d.geometry.TriangleMesh()
mesh.vertices = o3d.utility.Vector3dVector(vertices.detach().cpu().numpy().astype(np.float64, copy=False))
mesh.triangles = o3d.utility.Vector3iVector(faces.detach().cpu().numpy().astype(np.int32, copy=False))

voxel_size_scalar = float(voxel_size[0].item())
voxel_grid = o3d.geometry.VoxelGrid.create_from_triangle_mesh_within_bounds(
    mesh,
    voxel_size=voxel_size_scalar,
    min_bound=AABB[0].detach().cpu().numpy().astype(np.float64, copy=False),
    max_bound=AABB[1].detach().cpu().numpy().astype(np.float64, copy=False),
)

o3d_voxels = voxel_grid.get_voxels()
o3d_ijk_np = np.asarray([v.grid_index for v in o3d_voxels], dtype=np.int32).reshape(-1, 3)
o3d_ijk = torch.from_numpy(o3d_ijk_np).to(device=device, dtype=torch.int32)
o3d_unique = torch.unique(o3d_ijk, dim=0) if o3d_ijk.numel() > 0 else torch.empty((0, 3), device=device, dtype=torch.int32)

print('open3d unique voxel count:', o3d_unique.shape[0])


open3d unique voxel count: 1049317


In [8]:
gx, gy, gz = [int(x) for x in (grid_range[1] - grid_range[0]).tolist()]

ours_keys = (
    ours_unique[:, 0].to(torch.int64)
    + gx * (ours_unique[:, 1].to(torch.int64) + gy * ours_unique[:, 2].to(torch.int64))
)
o3d_keys = (
    o3d_unique[:, 0].to(torch.int64)
    + gx * (o3d_unique[:, 1].to(torch.int64) + gy * o3d_unique[:, 2].to(torch.int64))
)

ours_only_mask = ~torch.isin(ours_keys, o3d_keys)
o3d_only_mask = ~torch.isin(o3d_keys, ours_keys)
ours_only = ours_unique[ours_only_mask]
o3d_only = o3d_unique[o3d_only_mask]

num_ours = int(ours_unique.shape[0])
num_o3d = int(o3d_unique.shape[0])
num_ours_only = int(ours_only.shape[0])
num_o3d_only = int(o3d_only.shape[0])
num_intersection = num_ours - num_ours_only
num_union = num_intersection + num_ours_only + num_o3d_only
jaccard = float(num_intersection / num_union) if num_union > 0 else 1.0

print('ours:', num_ours)
print('open3d:', num_o3d)
print('intersection:', num_intersection)
print('only ours:', num_ours_only)
print('only open3d:', num_o3d_only)
print('union:', num_union)
print('jaccard:', jaccard)


ours: 1049126
open3d: 1049317
intersection: 1049126
only ours: 0
only open3d: 183
union: 1049309
jaccard: 0.9998255995135846


In [9]:
print('sample only-ours voxels:')
print(ours_only[:50].detach().cpu())
print('sample only-open3d voxels:')
print(o3d_only[:50].detach().cpu())


sample only-ours voxels:
tensor([], size=(0, 3), dtype=torch.int32)
sample only-open3d voxels:
tensor([[ 27, 128,  68],
        [ 29, 128,  57],
        [ 39, 128,  44],
        [ 41,  23, 128],
        [ 47, 128,  55],
        [ 48, 128,  40],
        [ 51,  64, 128],
        [ 52,  36, 128],
        [ 52, 128,  80],
        [ 54,  96, 128],
        [ 57, 128,  23],
        [ 58,  42, 128],
        [ 58,  88, 128],
        [ 58, 128,  49],
        [ 63, 128,  85],
        [ 64, 127, 128],
        [ 64, 128, 127],
        [ 64, 128, 128],
        [ 66, 128,  83],
        [ 68,  82, 128],
        [ 71, 128,  49],
        [ 72,  71, 128],
        [ 72,  81, 128],
        [ 73,  85, 128],
        [ 75,  52, 128],
        [ 76,  49, 128],
        [ 81,  68, 128],
        [ 81, 128,  71],
        [ 82,  52, 128],
        [ 83, 128,  50],
        [ 86, 128,  53],
        [ 89, 128,  89],
        [ 91, 128,  69],
        [ 94, 128,  87],
        [ 99, 128,  82],
        [100,  64, 128],
     